In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, math, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split

ROOT = Path('/content/drive/MyDrive/TrafficAI_ControlCenter')
DATA = ROOT / 'datasets' / 'processed' / 'module1_bus' / 'module1_ml_table.parquet'
MODEL_DIR = ROOT / 'models' / 'module1' / 'dl'
MET_DIR = ROOT / 'metrics' / 'module1'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MET_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda')
df = pd.read_parquet(DATA)
print(df['priority_action'].value_counts())
print('rows', len(df))

Mounted at /content/drive
priority_action
NONE            415421
TRUNCATE_RED      3044
EXTEND_GREEN       988
Name: count, dtype: int64
rows 419453


In [2]:
feat_cols = ['dist_to_junction_m','speed_mps','eta_sec','lateness_sec','t_sec','direction_up','node','route_code']
df = df.sort_values(['route_code','bus_number','month','day','direction','t_sec','node']).copy()
df[feat_cols] = df[feat_cols].fillna(df[feat_cols].median(numeric_only=True))

SEQ_LEN = 16
groups = ['route_code','bus_number','month','day','direction']

Xs, ys = [], []
for _, g in df.groupby(groups, sort=False):
    if len(g) < SEQ_LEN:
        continue
    arr = g[feat_cols].to_numpy(dtype=np.float32)
    labels = g['y'].to_numpy(dtype=np.int64)
    # sliding windows; label = last step action
    for i in range(SEQ_LEN-1, len(g)):
        Xs.append(arr[i-SEQ_LEN+1:i+1])
        ys.append(labels[i])

X = np.stack(Xs)   # [N, T, F]
y = np.array(ys)
print('sequences', X.shape, 'label counts', np.bincount(y, minlength=3))

# standardize with train stats only
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
mean = X_train.mean(axis=(0,1), keepdims=True)
std = X_train.std(axis=(0,1), keepdims=True) + 1e-6
X_train = (X_train - mean) / std
X_test = (X_test - mean) / std
np.save(MODEL_DIR / 'seq_mean.npy', mean)
np.save(MODEL_DIR / 'seq_std.npy', std)

sequences (335434, 16, 8) label counts [331448    979   3007]


In [3]:
class SeqDS(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X)
        self.y = torch.tensor(y)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

train_loader = DataLoader(SeqDS(X_train, y_train), batch_size=256, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(SeqDS(X_test, y_test), batch_size=512, shuffle=False, num_workers=2, pin_memory=True)

# class-weighted loss (priority events are rare)
counts = np.bincount(y_train, minlength=3).astype(np.float32)
weights = counts.sum() / (len(counts) * np.maximum(counts, 1))
class_weights = torch.tensor(weights, dtype=torch.float32, device=device)
print('class_weights', class_weights.tolist())

def evaluate(model):
    model.eval()
    preds, trues, probas = [], [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(device)
            logits = model(xb)
            prob = torch.softmax(logits, dim=1).cpu().numpy()
            pred = prob.argmax(1)
            preds.append(pred)
            trues.append(yb.numpy())
            probas.append(prob)
    y_true = np.concatenate(trues)
    y_pred = np.concatenate(preds)
    y_prob = np.concatenate(probas)
    out = {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'precision': float(precision_score(y_true, y_pred, average='macro', zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, average='macro', zero_division=0)),
        'f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
    }
    try:
        out['roc_auc'] = float(roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro'))
    except Exception:
        out['roc_auc'] = None
    return out

def train_model(model, name, epochs=8, lr=1e-3):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss(weight=class_weights)
    best_f1, best_state = -1, None
    for ep in range(1, epochs+1):
        model.train()
        total = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            opt.step()
            total += loss.item() * len(yb)
        metrics = evaluate(model)
        print(f'{name} epoch {ep}/{epochs} loss={total/len(y_train):.4f} f1={metrics["f1"]:.4f} acc={metrics["accuracy"]:.4f}')
        if metrics['f1'] > best_f1:
            best_f1 = metrics['f1']
            best_state = {k: v.detach().cpu().clone() for k,v in model.state_dict().items()}
    model.load_state_dict(best_state)
    metrics = evaluate(model)
    metrics.update({'model': name, 'category': 'Optimized DL', 'module': 'module1_bus'})
    torch.save({'state_dict': best_state, 'feat_cols': feat_cols, 'seq_len': SEQ_LEN}, MODEL_DIR / f'{name}.pt')
    (MET_DIR / f'{name}.json').write_text(json.dumps(metrics, indent=2))
    print('Saved', name, metrics)
    return metrics

class_weights [0.3373422622680664, 114.23882293701172, 37.17747116088867]


In [5]:
F = len(feat_cols)  # 8
C = 3

class AttnPool(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.w = nn.Linear(d, 1)
    def forward(self, h):  # [B,T,D]
        a = torch.softmax(self.w(h), dim=1)
        return (a * h).sum(dim=1)

# 4 LSTM + Attention
class LSTMAttn(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(F, 64, batch_first=True, bidirectional=False)
        self.attn = AttnPool(64)
        self.fc = nn.Linear(64, C)
    def forward(self, x):
        h,_ = self.lstm(x)
        return self.fc(self.attn(h))

# 5 BiLSTM + Attention
class BiLSTMAttn(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(F, 64, batch_first=True, bidirectional=True)
        self.attn = AttnPool(128)
        self.fc = nn.Linear(128, C)
    def forward(self, x):
        h,_ = self.lstm(x)
        return self.fc(self.attn(h))

# 6 TCN + Attention (simple temporal conv stack)
class TCNAttn(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(F, 64, kernel_size=3, padding=2, dilation=1), nn.ReLU(),
            nn.Conv1d(64, 64, kernel_size=3, padding=4, dilation=2), nn.ReLU(),
            nn.Conv1d(64, 64, kernel_size=3, padding=8, dilation=4), nn.ReLU(),
        )
        self.attn = AttnPool(64)
        self.fc = nn.Linear(64, C)
    def forward(self, x):
        z = self.net(x.transpose(1,2)).transpose(1,2)[:, -SEQ_LEN:, :]
        z = z[:, -x.size(1):, :]
        return self.fc(self.attn(z))

# 7 Transformer + Feature Attention
class FeatureAttn(nn.Module):
    def __init__(self):
        super().__init__()
        self.feat = nn.Sequential(nn.Linear(F, 64), nn.ReLU(), nn.Linear(64, F), nn.Sigmoid())
        enc = nn.TransformerEncoderLayer(d_model=64, nhead=4, dim_feedforward=128, batch_first=True)
        self.proj = nn.Linear(F, 64)
        self.encoder = nn.TransformerEncoder(enc, num_layers=2)
        self.attn = AttnPool(64)
        self.fc = nn.Linear(64, C)
    def forward(self, x):
        # feature attention on last step broadcast
        fa = self.feat(x[:, -1, :]).unsqueeze(1)
        x = x * fa
        h = self.encoder(self.proj(x))
        return self.fc(self.attn(h))

# 8 Transformer Encoder + class-weighted learning (weight is in loss already)
class TransformerCW(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Linear(F, 64)
        layer = nn.TransformerEncoderLayer(d_model=64, nhead=4, dim_feedforward=128, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=3)
        self.fc = nn.Linear(64, C)
    def forward(self, x):
        h = self.encoder(self.proj(x))
        return self.fc(h[:, -1, :])

# 9 GAT + GRU  (lightweight graph: each timestep = node, edges to neighbors)
class GATGRU(nn.Module):
    def __init__(self):
        super().__init__()
        self.in_proj = nn.Linear(F, 32)
        self.attn_l = nn.Linear(64, 1)
        self.gru = nn.GRU(32, 64, batch_first=True)
        self.fc = nn.Linear(64, C)
    def forward(self, x):
        # x: [B,T,F]
        b,t,f = x.shape
        h = torch.relu(self.in_proj(x))  # [B,T,32]
        # simple self-graph attention over time
        # score between t and neighbors via concat
        out = []
        for i in range(t):
            center = h[:, i:i+1, :].expand(-1, t, -1)
            score = self.attn_l(torch.cat([center, h], dim=-1)).squeeze(-1)  # [B,T]
            # mask far nodes lightly
            idx = torch.arange(t, device=x.device)
            dist = (idx - i).abs().float()
            score = score - (dist.unsqueeze(0) / max(t-1,1))
            a = torch.softmax(score, dim=1).unsqueeze(-1)
            out.append((a * h).sum(dim=1))
        g = torch.stack(out, dim=1)  # [B,T,32]
        y,_ = self.gru(g)
        return self.fc(y[:, -1, :])

# 10 Decision / multi-task policy head on fused embedding
class PolicyHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = BiLSTMAttn()
        # replace final fc with deeper policy head
        self.encoder.fc = nn.Identity()
        self.policy = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, C)
        )
    def forward(self, x):
        z = self.encoder(x)
        return self.policy(z)

In [6]:
class TCNAttn(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(F, 64, 3, padding=1)
        self.conv2 = nn.Conv1d(64, 64, 3, padding=1)
        self.conv3 = nn.Conv1d(64, 64, 3, padding=1)
        self.attn = AttnPool(64)
        self.fc = nn.Linear(64, C)
    def forward(self, x):
        z = x.transpose(1, 2)
        z = torch.relu(self.conv1(z))
        z = torch.relu(self.conv2(z))
        z = torch.relu(self.conv3(z))
        z = z.transpose(1, 2)
        return self.fc(self.attn(z))

In [7]:
results = []
results.append(train_model(LSTMAttn(), 'LSTM_Attention', epochs=8))
results.append(train_model(BiLSTMAttn(), 'BiLSTM_Attention', epochs=8))
results.append(train_model(TCNAttn(), 'TCN_Attention', epochs=8))
results.append(train_model(FeatureAttn(), 'Transformer_Feature_Attention', epochs=8))
results.append(train_model(TransformerCW(), 'Transformer_Encoder_ClassWeighted', epochs=8))
results.append(train_model(GATGRU(), 'GAT_GRU', epochs=6))          # a bit heavier
results.append(train_model(PolicyHead(), 'Decision_Policy_Head', epochs=8))

sum_df = pd.DataFrame(results)[['model','category','accuracy','precision','recall','f1','roc_auc']]
print(sum_df.sort_values('f1', ascending=False))
sum_df.to_csv(MET_DIR / 'dl_summary.csv', index=False)
(MET_DIR / 'dl_summary.json').write_text(json.dumps(results, indent=2))

# merge with baselines for arena table
base = json.loads((MET_DIR / 'baselines_summary.json').read_text())
arena = base + results
pd.DataFrame(arena).to_csv(MET_DIR / 'arena_module1_11models.csv', index=False)
(MET_DIR / 'arena_module1_11models.json').write_text(json.dumps(arena, indent=2))
print('STEP 5b OK — Module 1 eleven-model arena saved')

LSTM_Attention epoch 1/8 loss=0.3995 f1=0.6742 acc=0.9648
LSTM_Attention epoch 2/8 loss=0.1160 f1=0.6817 acc=0.9692
LSTM_Attention epoch 3/8 loss=0.0732 f1=0.8222 acc=0.9886
LSTM_Attention epoch 4/8 loss=0.0554 f1=0.8183 acc=0.9885
LSTM_Attention epoch 5/8 loss=0.0441 f1=0.7398 acc=0.9806
LSTM_Attention epoch 6/8 loss=0.0409 f1=0.7878 acc=0.9881
LSTM_Attention epoch 7/8 loss=0.0333 f1=0.8459 acc=0.9920
LSTM_Attention epoch 8/8 loss=0.0341 f1=0.8951 acc=0.9944
Saved LSTM_Attention {'accuracy': 0.9943655253625889, 'precision': 0.8253846032245956, 'recall': 0.9953044034425597, 'f1': 0.8950576667995277, 'roc_auc': 0.9997565986645341, 'model': 'LSTM_Attention', 'category': 'Optimized DL', 'module': 'module1_bus'}
BiLSTM_Attention epoch 1/8 loss=0.2595 f1=0.6381 acc=0.9714
BiLSTM_Attention epoch 2/8 loss=0.0695 f1=0.7591 acc=0.9853
BiLSTM_Attention epoch 3/8 loss=0.0639 f1=0.7282 acc=0.9817
BiLSTM_Attention epoch 4/8 loss=0.0433 f1=0.7403 acc=0.9846
BiLSTM_Attention epoch 5/8 loss=0.0758 f1=